# Libs: 10 фабричных smoke-тестов без сети

Каждый путь создаёт объект нужной роли и кэширует его.

In [ ]:
from pathlib import Path
while not Path('.zemicomp').is_file():
    %cd ..

In [ ]:
import socket
from zemi.arsenal.libs import Libs
libs = Libs('http://127.0.0.1:8080/v1', model='test-model', context_window=8192, timeout=5.0)
paths = {'openai':'client','litellm':'router','dspy':'model','instructor':'client','pydantic_ai':'model','smolagents':'model','llama_index':'model','httpx':'client','outlines':'model','guidance':'model'}
assert tuple(paths) == libs.names and len(paths) == 10

In [ ]:
old_connect = socket.socket.connect
try:
    socket.socket.connect = lambda *_a, **_k: (_ for _ in ()).throw(AssertionError('factory attempted network access'))
    results = {}
    for library, role in paths.items():
        adapter = getattr(libs, library)
        value = getattr(adapter, role)
        assert value is getattr(adapter, role)
        results[f'libs.{library}.{role}'] = f'{type(value).__module__}.{type(value).__name__}'
finally:
    socket.socket.connect = old_connect
results